# Visualización de la Evaluación de Métodos de Extracción
Este notebook consolida los resultados de todos los métodos evaluados y permite comparar visualmente su rendimiento en los archivos de `extract-bench`, diferenciando entre imágenes de la carpeta `images` y texto seleccionable de la carpeta `pdfs`.

In [1]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Configurar estilo de Seaborn
sns.set_theme(style="whitegrid")

# Configurar fuente Palatino
plt.rcParams['font.family'] = 'serif'

### 1. Carga de Datos
Iteramos sobre todos los archivos `results.csv` en la carpeta `evaluations_result/` para crear un único DataFrame, diferenciando entre `images` (imágenes) y `pdfs` (texto seleccionable).

In [ ]:
eval_dir = Path("evaluations_result")
all_data = []

for doc_type in ["images", "pdfs"]:
    type_dir = eval_dir / doc_type
    if not type_dir.exists():
        continue
    for method_dir in type_dir.iterdir():
        if method_dir.is_dir():
            csv_path = method_dir / "results.csv"
            if csv_path.exists():
                df = pd.read_csv(csv_path)
                df['Metodo'] = method_dir.name
                df['Tipo Documento'] = doc_type
                all_data.append(df)

if all_data:
    df_full = pd.concat(all_data, ignore_index=True)
    # Limpiar errores: Convertir "Error" a NaN y asegurar que la columna es numérica
    df_full['Puntuación'] = pd.to_numeric(df_full['Puntuación'], errors='coerce')
    df_full = df_full.dropna(subset=['Puntuación'])
    print(f"Cargados {len(df_full)} registros de evaluación en total.")
    
    # Separar por tipo de documento
    df_images = df_full[df_full['Tipo Documento'] == 'images'].copy()
    df_pdfs = df_full[df_full['Tipo Documento'] == 'pdfs'].copy()
    
    # Limpiar nombre de archivo: quitar extensión para mostrar solo el stem
    df_images['Archivo'] = df_images['Archivo'].str.replace(r'\.(pdf|image)$', '', regex=True)
    df_pdfs['Archivo'] = df_pdfs['Archivo'].str.replace(r'\.(pdf|image)$', '', regex=True)
    
    mean_scores_images = df_images.groupby(['Metodo', 'Tipo Documento'])['Puntuación'].mean().reset_index()
    mean_scores_pdfs = df_pdfs.groupby(['Metodo', 'Tipo Documento'])['Puntuación'].mean().reset_index()
    
    method_colors = {
        'Docling': '#4A90D9',
        'Docling + header': '#2C5F8E',
        'MinerU': '#50B872',
        'MinerU + header': '#2E8B57',
        'PyMuPDF4LLM': '#E8833A',
        'PyMuPDF4LLM + header': '#B5622A',
        'PyMuPDF': '#9B72CF',
        'PyMuPDF + header': '#6B4FA0'
    }
else:
    print("No se encontraron resultados.")
    df_images = pd.DataFrame()
    df_pdfs = pd.DataFrame()

### 2. Imágenes - Puntuación Media por Método

In [3]:
if not df_images.empty:
    plt.figure(figsize=(10, 6))
    
    colors = [method_colors[m] for m in mean_scores_images['Metodo'].unique()]
    
    ax = sns.barplot(data=mean_scores_images, x='Puntuación', y='Metodo', hue='Metodo', palette=colors, dodge=False, legend=False)
    plt.title('Puntuación Media por Método - Imágenes', fontsize=16)
    plt.xlabel('Puntuación Media', fontsize=12)
    plt.ylabel('Método', fontsize=12)
    
    ax.grid(axis='both', alpha=0.2)
    
    for p in ax.patches:
        width = p.get_width()
        plt.text(width + 0.01, p.get_y() + p.get_height()/2. + 0.05, '{:1.3f}'.format(width), ha="left")
    
    plt.xlim(0, 1.1)
    plt.tight_layout()
    plt.show()
else:
    print("No hay datos de imágenes para visualizar.")

### 3. Imágenes - Distribución de Puntuaciones (Boxplot)

In [4]:
if not df_images.empty:
    plt.figure(figsize=(10, 6))
    
    colors = [method_colors[m] for m in df_images['Metodo'].unique()]
    
    sns.boxplot(data=df_images, x='Puntuación', y='Metodo', hue='Metodo', palette=colors, dodge=False, legend=False)
    sns.stripplot(data=df_images, x='Puntuación', y='Metodo', color='black', alpha=0.3, size=4, jitter=True)
    
    plt.grid(axis='both', alpha=0.2)
    
    plt.title('Distribución de Puntuaciones por Método - Imágenes', fontsize=16)
    plt.xlabel('Puntuación', fontsize=12)
    plt.ylabel('Método', fontsize=12)
    
    plt.tight_layout()
    plt.show()
else:
    print("No hay datos de imágenes para visualizar.")

### 4. PDFs (Texto Seleccionable) - Puntuación Media por Método

In [5]:
if not df_pdfs.empty:
    plt.figure(figsize=(10, 6))
    
    colors = [method_colors[m] for m in mean_scores_pdfs['Metodo'].unique()]
    
    ax = sns.barplot(data=mean_scores_pdfs, x='Puntuación', y='Metodo', hue='Metodo', palette=colors, dodge=False, legend=False)
    plt.title('Puntuación Media por Método - PDFs (Texto Seleccionable)', fontsize=16)
    plt.xlabel('Puntuación Media', fontsize=12)
    plt.ylabel('Método', fontsize=12)
    
    ax.grid(axis='both', alpha=0.2)
    
    for p in ax.patches:
        width = p.get_width()
        plt.text(width + 0.01, p.get_y() + p.get_height()/2. + 0.05, '{:1.3f}'.format(width), ha="left")
    
    plt.xlim(0, 1.1)
    plt.tight_layout()
    plt.show()
else:
    print("No hay datos de PDFs para visualizar.")

### 5. PDFs (Texto Seleccionable) - Distribución de Puntuaciones (Boxplot)
Muestra la dispersión: si las cajas son pequeñas significa que el modelo es muy consistente. Si hay muchos puntos fuera (outliers) o las cajas son largas, significa que a veces saca 1 y otras 0.

In [6]:
if not df_pdfs.empty:
    plt.figure(figsize=(10, 6))
    
    colors = [method_colors[m] for m in df_pdfs['Metodo'].unique()]
    
    sns.boxplot(data=df_pdfs, x='Puntuación', y='Metodo', hue='Metodo', palette=colors, dodge=False, legend=False)
    sns.stripplot(data=df_pdfs, x='Puntuación', y='Metodo', color='black', alpha=0.3, size=4, jitter=True)
    
    plt.grid(axis='both', alpha=0.2)
    
    plt.title('Distribución de Puntuaciones por Método - PDFs (Texto Seleccionable)', fontsize=16)
    plt.xlabel('Puntuación', fontsize=12)
    plt.ylabel('Método', fontsize=12)
    
    plt.tight_layout()
    plt.show()
else:
    print("No hay datos de PDFs para visualizar.")